# 05 Kafka Producer fuer Wetterdaten

Dieses Notebook implementiert BD-14. Es liest die bereinigten Wetterdaten aus `data/silver/weather_features.parquet`, serialisiert jede Match-Wetterbeobachtung als JSON-Message und sendet sie an das Kafka Topic `weather.observations`.

Die Kafka-Verbindung wird ueber `KAFKA_BOOTSTRAP_SERVERS` konfiguriert. Dadurch kann derselbe Code gegen eine lokale Kafka-Instanz oder gegen einen bereitgestellten Kafka-Broker im Cluster ausgefuehrt werden.

## Methodischer Ansatz

Producer, Topic und Consumer werden als getrennte Pipeline-Schritte behandelt. Dieses Notebook uebernimmt ausschliesslich den Producer-Schritt: Wetterdaten werden aus der Silver-Schicht gelesen, als JSON serialisiert und mit einem stabilen Message-Key an Kafka gesendet.

Topic, Message-Key und JSON-Schema sind in `docs/kafka_topics.md` definiert. Spark liest und parst die Messages im nachfolgenden Kafka-Read-Schritt.

In [ ]:
import json
import math
import os
import time

import pandas as pd

from project_paths import project_root as get_project_root
from pipeline_config import DEFAULT_KAFKA_BOOTSTRAP_SERVERS, DEFAULT_KAFKA_WEATHER_TOPIC, WEATHER_SCHEMA_VERSION

try:
    from confluent_kafka import Producer
    from confluent_kafka.admin import AdminClient, NewTopic
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'Missing dependency confluent-kafka. Install project dependencies with '
        '`pip install -r /home/jovyan/work/requirements.txt` in the Jupyter environment.'
    ) from exc

KAFKA_BOOTSTRAP_SERVERS = os.getenv('KAFKA_BOOTSTRAP_SERVERS', DEFAULT_KAFKA_BOOTSTRAP_SERVERS)
KAFKA_TOPIC = os.getenv('KAFKA_WEATHER_TOPIC', DEFAULT_KAFKA_WEATHER_TOPIC)
KAFKA_RESET_WEATHER_TOPIC = os.getenv('KAFKA_RESET_WEATHER_TOPIC', 'false').lower() in {'1', 'true', 'yes'}
SCHEMA_VERSION = WEATHER_SCHEMA_VERSION

{
    'bootstrap_servers': KAFKA_BOOTSTRAP_SERVERS,
    'topic': KAFKA_TOPIC,
    'reset_topic_before_send': KAFKA_RESET_WEATHER_TOPIC,
}


## Wetterdaten laden und Schema pruefen

`match_id` ist der Kafka-Key, weil er pro Wetterbeobachtung eindeutig ist und fuer Deduplizierung und Joins genutzt wird. `weather_lookup_key` bleibt im JSON-Value erhalten, ist aber nicht eindeutig pro Match.

In [ ]:
project_root = get_project_root()
weather_input_path = project_root / 'data' / 'silver' / 'weather_features.parquet'
tables_output_path = project_root / 'outputs' / 'tables'
tables_output_path.mkdir(parents=True, exist_ok=True)

weather_features = pd.read_parquet(weather_input_path)

required_columns = [
    'match_id', 'match_date', 'kick_off', 'kickoff_timestamp', 'weather_time',
    'hours_from_kickoff', 'weather_lookup_key', 'stadium_id', 'stadium_name',
    'city_name', 'country_name', 'latitude', 'longitude', 'openmeteo_timezone',
    'temperature_c', 'feels_like_c', 'precipitation_mm', 'rain_mm', 'rain_flag',
    'temperature_group', 'weather_missing_flag', 'tournament_short_name',
    'competition_name', 'season_name'
]

missing_columns = sorted(set(required_columns) - set(weather_features.columns))
assert not missing_columns, f'Missing required weather columns: {missing_columns}'
assert not weather_features.empty, 'Weather feature dataset must not be empty.'
assert weather_features['match_id'].is_unique, 'match_id must be unique for Kafka message keys.'

weather_features = weather_features.sort_values('match_id').reset_index(drop=True)

{
    'input_path': str(weather_input_path.relative_to(project_root)),
    'rows': len(weather_features),
    'unique_match_ids': int(weather_features['match_id'].nunique()),
    'unique_weather_lookup_keys': int(weather_features['weather_lookup_key'].nunique()),
    'temperature_groups': weather_features['temperature_group'].value_counts(dropna=False).to_dict(),
    'rainy_matches': int(weather_features['rain_flag'].sum()),
}


## JSON-Messages bauen

Die JSON-Felder folgen `docs/kafka_topics.md`. Zeitstempel werden als ISO-8601-Strings geschrieben, damit Spark sie in BD-15 mit einem festen Schema parsen kann.

In [ ]:
message_fields = [
    'match_id', 'match_date', 'kick_off', 'kickoff_timestamp', 'weather_time',
    'hours_from_kickoff', 'weather_lookup_key', 'stadium_id', 'stadium_name',
    'city_name', 'country_name', 'latitude', 'longitude', 'openmeteo_timezone',
    'temperature_c', 'feels_like_c', 'precipitation_mm', 'rain_mm', 'rain_flag',
    'temperature_group', 'weather_missing_flag', 'tournament_short_name',
    'competition_name', 'season_name'
]


def to_jsonable(value):
    if pd.isna(value):
        return None
    if hasattr(value, 'isoformat'):
        return value.isoformat()
    if hasattr(value, 'item'):
        return value.item()
    if isinstance(value, float) and math.isnan(value):
        return None
    return value


def build_weather_message(row: pd.Series) -> dict:
    message = {field: to_jsonable(row[field]) for field in message_fields}
    message['match_id'] = int(message['match_id'])
    message['stadium_id'] = int(message['stadium_id'])
    message['rain_flag'] = bool(message['rain_flag'])
    message['weather_missing_flag'] = bool(message['weather_missing_flag'])
    message['schema_version'] = SCHEMA_VERSION
    return message


messages = []
for _, row in weather_features.iterrows():
    message = build_weather_message(row)
    key = str(message['match_id'])
    value = json.dumps(message, ensure_ascii=False, sort_keys=True)
    messages.append({'key': key, 'value': value, 'message': message})

sample_record = messages[0]
print('Sample key:', sample_record['key'])
print(json.dumps(sample_record['message'], ensure_ascii=False, indent=2, sort_keys=True))

## Topic pruefen und Messages senden

Der AdminClient erstellt das Topic bei Bedarf. Der Producer sendet danach eine Message pro Wetterzeile. Der Delivery Callback zaehlt erfolgreiche und fehlerhafte Sendungen.

In [ ]:
admin = AdminClient({'bootstrap.servers': KAFKA_BOOTSTRAP_SERVERS})
metadata = admin.list_topics(timeout=10)
topic_was_reset = False

if KAFKA_RESET_WEATHER_TOPIC and KAFKA_TOPIC in metadata.topics:
    futures = admin.delete_topics([KAFKA_TOPIC], operation_timeout=30)
    for topic, future in futures.items():
        future.result(timeout=30)
    topic_was_reset = True

    for _ in range(10):
        metadata = admin.list_topics(timeout=10)
        if KAFKA_TOPIC not in metadata.topics:
            break
        time.sleep(1)
else:
    metadata = admin.list_topics(timeout=10)

if KAFKA_TOPIC not in metadata.topics:
    futures = admin.create_topics([
        NewTopic(KAFKA_TOPIC, num_partitions=1, replication_factor=1)
    ])
    for topic, future in futures.items():
        future.result(timeout=30)

producer = Producer({
    'bootstrap.servers': KAFKA_BOOTSTRAP_SERVERS,
    'client.id': 'football-weather-bd14-producer',
})

delivered = []
delivery_errors = []


def delivery_report(error, msg):
    if error is not None:
        delivery_errors.append(str(error))
    else:
        delivered.append({
            'topic': msg.topic(),
            'partition': msg.partition(),
            'offset': msg.offset(),
            'key': msg.key().decode('utf-8') if msg.key() else None,
        })


for record in messages:
    producer.produce(
        KAFKA_TOPIC,
        key=record['key'].encode('utf-8'),
        value=record['value'].encode('utf-8'),
        callback=delivery_report,
    )
    producer.poll(0)

remaining = producer.flush(timeout=60)

status = {
    'topic': KAFKA_TOPIC,
    'bootstrap_servers': KAFKA_BOOTSTRAP_SERVERS,
    'reset_requested': KAFKA_RESET_WEATHER_TOPIC,
    'topic_was_reset': topic_was_reset,
    'input_rows': len(weather_features),
    'messages_prepared': len(messages),
    'messages_delivered': len(delivered),
    'delivery_errors': len(delivery_errors),
    'remaining_after_flush': remaining,
    'sample_key': sample_record['key'],
    'sample_value_json': sample_record['value'],
}

pd.DataFrame([status]).to_csv(tables_output_path / 'kafka_weather_producer_status.csv', index=False)
pd.DataFrame([sample_record['message']]).to_csv(tables_output_path / 'kafka_weather_sample.csv', index=False)

assert remaining == 0, f'{remaining} messages remained unsent after flush.'
assert not delivery_errors, delivery_errors[:5]
assert len(delivered) == len(messages), f"Delivered {len(delivered)} of {len(messages)} messages."

status


## Ergebnis

BD-14 ist erfuellt, wenn die Statuszelle ohne Assertion-Fehler durchlaeuft und folgende Artefakte existieren:

- `outputs/tables/kafka_weather_producer_status.csv`
- `outputs/tables/kafka_weather_sample.csv`

Der Consumer in BD-15 kann anschliessend `weather.observations` lesen und ueber `match_id` deduplizieren.